In [ ]:
import pandas as pd
import polars as pl
import numpy as np
import xgboost as xgb
from src import exploration_cleaning_methods as ecm
from src import prediction_methods as pm

**Men**

In [ ]:
m_train_df: pl.DataFrame = pl.read_parquet("../data/proccessed/m_train_data.parquet")
m_train_df.head(10)

In [ ]:
m_grid_df: pl.DataFrame = pm.create_carthesian_matchup_grid(m_train_df)

In [ ]:
m_regular_season_detailed_lf: pl.LazyFrame = pl.scan_csv("../data/raw/MRegularSeasonDetailedResults.csv")
m_regular_season_detailed_lf.head(10).collect

In [ ]:
m_prof_df: pl.DataFrame = ecm.create_team_season_profile(
    m_regular_season_detailed_lf.filter(pl.col("Season") == 2026)
).collect()

In [ ]:
del m_regular_season_detailed_lf

In [ ]:
m_infer_df: pl.DataFrame = (
    m_grid_df
    .join(
        m_prof_df, 
        left_on=["Season", "ATeamID"], 
        right_on=["Season", "TeamID"], 
        how="left"
    )
    .join(
        m_prof_df, 
        left_on=["Season", "BTeamID"], 
        right_on=["Season", "TeamID"], 
        how="left", 
        suffix="_B"
))

In [ ]:
base_cols: list[str] = [
    "TeamScore", "OpponentScore", "WinRatio", "Location", "TeamPOS", 
    "OpponentPOS", "OffEfficiency", "DefEfficiency", "TeamEFG", 
    "OpponentEFG", "TurnoverRate"
]

for col in base_cols:
    m_infer_df = m_infer_df.with_columns((pl.col(col) - pl.col(f"{col}_B")).alias(f"{col}_Diff"))

In [ ]:
del m_grid_df, m_prof_df

In [ ]:
m_model: xgb.Booster = pm.generate_model(m_train_df)
feat_cols: list[str] = [f"{col}_Diff" for col in base_cols]
m_sub: pl.DataFrame = pm.predict_and_create_submission_data(m_model, feat_cols, m_infer_df)
m_sub.head(10)

**Women**

In [ ]:
w_train_df: pl.DataFrame = pl.read_parquet("../data/proccessed/w_train_data.parquet")
w_train_df.head(10)

In [ ]:
w_grid_df: pl.DataFrame = pm.create_carthesian_matchup_grid(w_train_df)

In [ ]:
w_regular_season_detailed_lf: pl.LazyFrame = pl.scan_csv("../data/raw/WRegularSeasonDetailedResults.csv")
w_regular_season_detailed_lf.head(10).collect

In [ ]:
w_prof_df: pl.DataFrame = ecm.create_team_season_profile(
    w_regular_season_detailed_lf.filter(pl.col("Season") == 2026)
).collect()

In [ ]:
del w_regular_season_detailed_lf

In [ ]:
w_infer_df: pl.DataFrame = (
    w_grid_df
    .join(
        w_prof_df, 
        left_on=["Season", "ATeamID"], 
        right_on=["Season", "TeamID"], 
        how="left"
    )
    .join(
        w_prof_df, 
        left_on=["Season", "BTeamID"], 
        right_on=["Season", "TeamID"], 
        how="left", 
        suffix="_B"
))

In [ ]:
base_cols: list[str] = [
    "TeamScore", "OpponentScore", "WinRatio", "Location", "TeamPOS", 
    "OpponentPOS", "OffEfficiency", "DefEfficiency", "TeamEFG", 
    "OpponentEFG", "TurnoverRate"
]

for col in base_cols:
    w_infer_df = w_infer_df.with_columns((pl.col(col) - pl.col(f"{col}_B")).alias(f"{col}_Diff"))

In [ ]:
del w_grid_df, base_cols, w_prof_df